# **CV融合算子实践**

通过前面章节的系统学习，我们已经掌握了使用 Ascend C 以 `<<<>>>` 直调方式开发 CV 融合算子的完整方法（参见 `02.08_cv_fused_operator_development.ipynb`）。为巩固所学知识，本节提供一个 **MatmulSinh** 融合算子的实践练习：以 `.asc` 直调工程的形式，独立完成 Host 侧 Tiling、Kernel 实现与 Host 调用代码，并编译运行验证。

融合算子将 Matmul、Bias 与 Sinh 激活融合为单一算子，计算公式为：

<div style="width: 40%; float: left; clear: both; margin: 10px 0;">

$$
c = \text{Sinh}\left(a \times b + bias\right)
$$
</div>
<div style="clear: both;"></div>

其中 $\text{Sinh}(x) = \dfrac{e^{x} - e^{-x}}{2}$。本实践中 `a`、`b`、`bias`、`c` 均为 `float` 类型，shape 固定为 `M = 1024`、`N = 2048`、`K = 512`。

算子原型：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" >matmul_sinh</td>
  </tr>
  <tr>
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(M, K) / (1024, 512)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(K, N) / (512, 2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(N) / (2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(M, N) / (1024, 2048)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

要求：
- 算子命名为 `matmul_sinh`，采用 `<<<>>>` 直调方式实现。
- 完成MatmulSinh算子kernel侧核函数实现相关代码补齐。
- 完成MatmulSinh算子host侧Tiling函数相关代码补齐。
- 支持 `float` 类型输入输出，shape 为 `M = 1024`、`N = 2048`、`K = 512`。

请开始你的实践，体验完整开发过程。

---

## **1. 环境准备**
本文所有内容均存放于 Sources 文件夹。在开始创建算子工程前，先要对 jupyter 环境进行初始化。以下代码完成了初始化并将环境中的变量导入 jupyter 环境，并完成代码目录的创建，保证能正常导入代码以及编译。

In [ ]:
!mkdir -p Sources/02.17

import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---
## **2. CV融合算子直调工程创建**
直调工程文件概述：
- `matmul_sinh.asc` 同时包含 Host 侧 tiling 生成、Kernel 实现和 Host 调用代码。
- `CMakeLists.txt` 负责把 `.asc` 文件编译链接成可执行程序。

开发者可以使用 CMake 作为构建系统，本实践完整的编译配置如下：

In [ ]:
%%writefile Sources/02.17/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, cpu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201")
set(TARGET_SRC "matmul_sinh.asc" CACHE STRING "ASC source file used to build demo")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    ${TARGET_SRC}
)

target_link_libraries(demo PRIVATE
    tiling_api
    register
    platform
    m
    dl
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)


---
## **3. CV融合算子代码实现**
`matmul_sinh.asc` 中完成 CV 融合算子的 Host 侧 Tiling、Kernel 实现与 Host 调用代码。MatmulSinh 融合算子的整体计算流程如下：

1. Matmul 计算数据搬入
2. Matmul 计算
3. Matmul 结果搬出
4. Sinh 计算数据搬入
5. Sinh 计算
6. 结果搬出

其中步骤 1~4（Matmul 计算数据搬入、Matmul 计算、Matmul 结果搬出和 Sinh 计算数据搬入）可以通过 Matmul API 直接实现。所以算子类包含以下函数：
- `MatmulCompute`：实现 Matmul 计算逻辑。
- `SinhCompute`：实现 Sinh 计算逻辑。由于硬件无直接的 Sinh 指令，需借助 `sinh(x) = (exp(x) - exp(-x)) * 0.5`，计算过程中可能需要一块临时缓存 `tmpBuf`。
- `CopyOut`：Sinh 计算结果搬出操作。由于输出是一小块矩阵，每完成一行小矩阵的填充后，需先跳过 `(整体矩阵单行元素数 - 小矩阵单行元素数)` 的地址偏移量，再执行下一行小矩阵结果的填充操作。
- `CalcOffset`：根据核编号按 M 方向遍历规则，计算当前核处理的 A/B/C/Bias 分块在 GM 上的起始偏移。
- `Init`：设置 GlobalTensor，并根据 `CalcOffset` 计算得到当前核处理的偏移。
- `Process`：设置当前核处理的 A/B/Bias 输入，并通过 `Iterate` 循环获取每次运算得到的 C 矩阵小块结果，用于 Sinh 计算。

下方代码保留了直调工程框架、数据准备和函数骨架，`TODO` 处需要学习者补全：Tiling 配置、GlobalTensor 初始化、Matmul 结果获取、Sinh 融合计算、结果搬出和分块偏移计算。

In [ ]:
%%writefile Sources/02.17/matmul_sinh.asc
#include <algorithm>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

using namespace matmul;

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            std::fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__,    \
                         __code);                                                                       \
            return 1;                                                                                   \
        }                                                                                               \
    } while (0)

AscendC::tiling::TCubeTiling GenerateTiling(platform_ascendc::PlatformAscendC* ascendcPlatform)
{
    constexpr int32_t M = 1024;
    constexpr int32_t N = 2048;
    constexpr int32_t K = 512;
    constexpr int32_t baseM = 128;
    constexpr int32_t baseN = 128;

    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);

    // TODO: 参照课程示例配置Matmul Tiling参数（本实践a/b/c/bias均为float类型）。
    // 需要完成的内容包括：SetDim、A/B/C/Bias类型、Shape、OrgShape、FixSplit、Bias、Traverse和BufferSpace。

    AscendC::tiling::TCubeTiling cubeTilingData;
    if (cubeTiling.GetTiling(cubeTilingData) == -1) {
        std::fprintf(stderr, "Generate matmul tiling failed.\n");
    }
    return cubeTilingData;
}

__aicore__ inline uint32_t Ceiling(uint32_t a, uint32_t b)
{
    return (a + b - 1) / b;
}


template <typename AType, typename BType, typename CType, typename BiasType>
class MatmulSinhKernel {
public:
    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
                                const TCubeTiling& tiling, AscendC::TPipe* pipe);
    __aicore__ inline void Process();
    __aicore__ inline void MatmulCompute();
    __aicore__ inline void SinhCompute();
    __aicore__ inline void CopyOut(uint32_t count);
    __aicore__ inline void CalcOffset(int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA,
                                      int32_t& offsetB, int32_t& offsetC, int32_t& offsetBias);

    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BType>,
           MatmulType<AscendC::TPosition::VECIN, CubeFormat::ND, CType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>>
        matmulObj;

    AscendC::GlobalTensor<AType> aGlobal;
    AscendC::GlobalTensor<BType> bGlobal;
    AscendC::GlobalTensor<CType> cGlobal;
    AscendC::GlobalTensor<BiasType> biasGlobal;
    AscendC::LocalTensor<CType> sinhOutLocal;
    AscendC::TBuf<AscendC::QuePosition::VECCALC> tmpBuf;  // Sinh计算所需的临时缓存
    TCubeTiling tiling;
    AscendC::TQue<AscendC::QuePosition::VECOUT, 1> sinhOutQueue;
};

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulSinhKernel<AType, BType, CType, BiasType>::Init(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c, const TCubeTiling& tiling,
    AscendC::TPipe* pipe)
{
    this->tiling = tiling;

    // TODO: 设置aGlobal、bGlobal、cGlobal、biasGlobal的GM地址和元素个数。
    // TODO: 调用CalcOffset计算当前逻辑核处理的A/B/C/Bias偏移，并更新GlobalTensor起始地址。
    // TODO: 初始化sinhOutQueue以及Sinh计算所需的tmpBuf缓存。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulSinhKernel<AType, BType, CType, BiasType>::Process()
{
    // TODO: 绑定当前核的A/B/Bias输入。
    // TODO: 通过Iterate循环获取每个baseM * baseN的Matmul结果，依次调用MatmulCompute、SinhCompute和CopyOut。
    // TODO: 结束Matmul对象。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulSinhKernel<AType, BType, CType, BiasType>::MatmulCompute()
{
    // TODO: 从sinhOutQueue申请LocalTensor，并通过GetTensorC获取Matmul中间结果。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulSinhKernel<AType, BType, CType, BiasType>::SinhCompute()
{
    // TODO: 对Matmul结果执行Sinh计算：sinh(x) = (exp(x) - exp(-x)) * 0.5。
    // 提示：借助tmpBuf.Get<CType>()申请临时Tensor，组合使用AscendC::Exp/Muls/Sub完成计算，
    //       最后将结果EnQue到sinhOutQueue。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulSinhKernel<AType, BType, CType, BiasType>::CopyOut(uint32_t count)
{
    // TODO: 从sinhOutQueue取出Sinh结果。
    // TODO: 按FIRSTM遍历顺序计算当前输出小块写回C矩阵的startOffset。
    // TODO: 使用DataCopyParams和DataCopy将小块结果搬出到GM。
    // TODO: 释放LocalTensor。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulSinhKernel<AType, BType, CType, BiasType>::CalcOffset(
    int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA, int32_t& offsetB, int32_t& offsetC,
    int32_t& offsetBias)
{
    // TODO: 根据blockIdx、singleCoreM、singleCoreN计算当前逻辑核对应的M/N方向分块索引。
    // TODO: 根据分块索引计算A、B、C、Bias在GM上的起始偏移。
}

extern "C" __global__ __mix__(1, 2) void matmul_sinh(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
    __gm__ uint8_t* __kfc_workspace__ workspace, AscendC::tiling::TCubeTiling tiling)
{
    AscendC::TPipe pipe;

    MatmulSinhKernel<float, float, float, float> op;
    op.Init(a, b, bias, c, tiling, &pipe);
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), op.matmulObj, &op.tiling);
    op.Process();
}

int32_t main(int32_t argc, char** argv)
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();

    constexpr uint32_t M = 1024;
    constexpr uint32_t N = 2048;
    constexpr uint32_t K = 512;
    constexpr uint32_t numBlocks = 1;
    constexpr size_t aFileSize = M * K * sizeof(float);
    constexpr size_t bFileSize = K * N * sizeof(float);
    constexpr size_t biasFileSize = N * sizeof(float);
    constexpr size_t cFileSize = M * N * sizeof(float);
    const size_t workspaceSize = static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    auto tiling = GenerateTiling(ascendcPlatform);

    constexpr int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t* inputADevice = nullptr;
    uint8_t* inputBDevice = nullptr;
    uint8_t* inputBiasDevice = nullptr;
    uint8_t* outputCDevice = nullptr;
    uint8_t* workspaceDevice = nullptr;
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputADevice), aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBDevice), bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBiasDevice), biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&outputCDevice), cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&workspaceDevice), workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // a * b + bias = 0.125 * 0.0625 * 512 + 0.5 = 4.5, sinh(4.5) = 45.00301...
    std::vector<float> inputAHost(M * K, 0.125f);
    std::vector<float> inputBHost(K * N, 0.0625f);
    std::vector<float> inputBiasHost(N, 0.5f);
    std::vector<float> outputCHost(M * N, 0.0f);
    const float goldenValue = std::sinh(4.5f);

    CHECK_ACL(aclrtMemcpy(inputADevice, aFileSize, inputAHost.data(), aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDevice, bFileSize, inputBHost.data(), bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDevice, biasFileSize, inputBiasHost.data(), biasFileSize,
                          ACL_MEMCPY_HOST_TO_DEVICE));

    matmul_sinh<<<numBlocks, 0, stream>>>(inputADevice, inputBDevice, inputBiasDevice, outputCDevice,
                                          workspaceDevice, tiling);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(outputCHost.data(), cFileSize, outputCDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));

    std::printf("result is:\n");
    for (uint32_t i = 0; i < 10; ++i) {
        std::printf("%.1f ", outputCHost[i]);
    }
    bool pass = true;
    for (size_t i = 0; i < outputCHost.size(); ++i) {
        if (std::fabs(outputCHost[i] - goldenValue) > 1e-2f) {
            pass = false;
            break;
        }
    }
    std::printf("\ntest %s\n", pass ? "pass" : "failed");

    CHECK_ACL(aclrtFree(inputADevice));
    CHECK_ACL(aclrtFree(inputBDevice));
    CHECK_ACL(aclrtFree(inputBiasDevice));
    CHECK_ACL(aclrtFree(outputCDevice));
    if (workspaceDevice != nullptr) {
        CHECK_ACL(aclrtFree(workspaceDevice));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}


---
## **4. CV融合算子编译运行**
补全 `matmul_sinh.asc` 中的 TODO 后，使用 `dav-2201` 架构编译直调工程并运行测试程序。这里以固定的 `M = 1024`、`N = 2048`、`K = 512` 进行测试，a 矩阵全 0.125，b 矩阵全 0.0625，bias 全 0.5，则 `a × b + bias = 0.125 × 0.0625 × 512 + 0.5 = 4.5`，`sinh(4.5) ≈ 45.0`。

In [ ]:
!rm -rf Sources/02.17/build
!bash -lc 'CANN_PATH=${ASCEND_HOME_PATH:-/usr/local/Ascend/cann}; source ${CANN_PATH}/set_env.sh && cmake -S Sources/02.17 -B Sources/02.17/build -DCMAKE_ASC_ARCHITECTURES=dav-2201 && cmake --build Sources/02.17/build -j && ./Sources/02.17/build/demo'

调用成功后，会有如下输出：
```
result is:
45.0 45.0 45.0 45.0 45.0 45.0 45.0 45.0 45.0 45.0
test pass
```

---
## **5. 查看答案**
实现方式不唯一，这里给出的答案仅供参考。

CMakeLists.txt：

In [ ]:
!cat ./answer/02.17_answer/CMakeLists.txt

matmul_sinh.asc：

In [ ]:
!cat ./answer/02.17_answer/matmul_sinh.asc